# 日温度 AutoReg 预测

使用日温度数据训练自回归模型，自动选择滞后阶数，并预测最后 30 天温度。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller


## 1. 读取数据


In [ ]:
temp = pd.read_csv(DATA_DIR / "daily_temperature.csv", parse_dates=["Date"])
series = temp.set_index("Date")["Temp"].asfreq("D")
series = series.interpolate(method="time").ffill().bfill()
train = series.iloc[:-30]
test = series.iloc[-30:]

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
train.plot(ax=ax, label="训练集")
test.plot(ax=ax, label="测试集")
ax.set_title("日温度序列")
ax.set_ylabel("温度")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

display(pd.concat({"train_tail": train.tail(), "test_head": test.head()}, axis=1))


## 2. 平稳性与相关性观察


In [ ]:
adf_stat, p_value, *_ = adfuller(train.dropna(), autolag="AIC")
ljung_box = acorr_ljungbox(train.dropna(), lags=[30], return_df=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
plot_acf(train, lags=60, ax=axes[0])
plot_pacf(train, lags=60, method="ywm", ax=axes[1])
axes[0].set_title("ACF")
axes[1].set_title("PACF")
plt.tight_layout()
plt.show()

display(
    pd.DataFrame(
        {
            "adf_stat": [adf_stat],
            "adf_p_value": [p_value],
            "ljung_box_p_value_lag30": [ljung_box["lb_pvalue"].iloc[0]],
        }
    ).round(4)
)


## 3. 使用 BIC 选择滞后阶数


In [ ]:
def select_autoreg_lag(values, max_lag=60):
    """Select the AutoReg lag order with the lowest BIC.

    Args:
        values: Training time series.
        max_lag: Largest lag order to evaluate.

    Returns:
        Data frame sorted by BIC with candidate lag orders.
    """
    records = []
    for lag in range(1, max_lag + 1):
        try:
            result = AutoReg(values, lags=lag, old_names=False).fit()
        except Exception:
            continue
        records.append({"lag": lag, "aic": result.aic, "bic": result.bic})
    if not records:
        raise ValueError("没有成功拟合任何 AutoReg 候选模型，请检查缺失值和 max_lag。")
    return pd.DataFrame(records).sort_values("bic").reset_index(drop=True)


lag_report = select_autoreg_lag(train, max_lag=60)
best_lag = int(lag_report.loc[0, "lag"])

print(f"推荐滞后阶数: {best_lag}")
display(lag_report.head(10).round(2))


## 4. 训练模型并预测


In [ ]:
model = AutoReg(train, lags=best_lag, old_names=False).fit()
forecast = model.predict(start=len(train), end=len(train) + len(test) - 1)
forecast.index = test.index

prediction = pd.DataFrame({"actual": test, "forecast": forecast})
prediction["error"] = prediction["actual"] - prediction["forecast"]

mae = mean_absolute_error(prediction["actual"], prediction["forecast"])
rmse = np.sqrt(mean_squared_error(prediction["actual"], prediction["forecast"]))
mape = (prediction["error"].abs() / prediction["actual"].abs()).mean()

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
series.loc["1990"].plot(ax=ax, label="实际值")
forecast.plot(ax=ax, label="预测值")
ax.axvline(test.index[0], color="gray", linestyle="--", alpha=0.8)
ax.set_title("AutoReg 日温度预测")
ax.set_ylabel("温度")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

display(prediction.round(2))
display(pd.DataFrame({"MAE": [mae], "RMSE": [rmse], "MAPE": [mape]}).round(4))
print(model.summary())
